# Optimise RSI Trailing Stop

Execute the stategy optimiser LLM agent to fine tune the parameters of the
RSI Trailing Stop strategy over a given time window.

### Setup Environment

In [ ]:
# Setup environment and depedencies
!poetry install

In [ ]:
# Load environment variables from .env file
%reload_ext dotenv
%dotenv ../.env

### Initialise Data Provider

In [ ]:
from datetime import datetime, timedelta
from qalc.market_data.providers import AlpacaDataApi  

# Set start and end dates for backtesting
end_date = datetime.now()
start_date = end_date - timedelta(days=60)

# Initialise the Alpaca market data provider
data_provider = AlpacaDataApi(symbol='AAPL')

### Display Backtest Bars

In [ ]:
from qalc.market_data.timeframe import TimeFrame

# Get the bars that will be used for the backtest
bars = data_provider.get_bars(start=start_date, end=end_date, 
                              timeframe=TimeFrame.FIFTEEN_MINUTES)
display(bars.head())

### Optimise Strategy

In [ ]:
from qalc.strategies import RSITrailingStop
from qalc.llm.agents import StrategyOptimiser

# Initialise the strategy
strategy = RSITrailingStop(
    trail_percent=0.03,
    rsi_period=14,
    rsi_threshold=30,
)

# Initialise LLM
from qalc.llm import llm_init
llm = llm_init(model="gpt-3.5-turbo", temperature=0.5)

# Initialis and run the strategy optimiser agent
agent = StrategyOptimiser(llm=llm, strategy=strategy, bars=bars)
results = agent.run(start_time=start_date, end_time=end_date)

display(results)

### Display Best Results

In [ ]:
# Extract and display the best parameters and results
best_params = results['best_parameters']
best_result = results['best_result']

print('Best Parameters:')
print(best_params)

print('Best Result:')
print(best_result)